# University of Engineering and Technology Peshawar, Nowshera Campus
Name: MUNSIFF ALI

Registration Number: 22JZELE0489
# Lab 12: LSTM for Time-Series Forecasting

**Topic:** Long Short-Term Memory network for AEP forecasting

## Lab Objective
The objective of this lab is to build and train an LSTM model for forecasting time-series values. LSTM layers are used because they can learn sequential dependencies across time steps.


## 1. Set Working Directory
The working directory is set so the notebook can access local datasets, helper modules, checkpoints, and saved training outputs.


In [1]:
import os
os.chdir(r'C:\Users\Admin\Downloads\labs\New folder\Code')

## 2. Import Libraries and Utilities
This cell imports forecasting metrics, sequence preparation functions, TensorFlow/Keras layers, callbacks, plotting tools, and scaling utilities.


In [2]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

## 3. Define Initial Parameters
The time-window length, number of features, starting epoch, and model variable are initialized before creating the LSTM network.


In [3]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

## 4. Define the LSTM Architecture
The LSTM model processes sequential input data using stacked LSTM layers, then outputs one predicted value through a dense layer.


In [4]:
def create_lstm():
    input_data = Input(shape=(time_steps, num_features))
    lstm_layer1 = LSTM(8, return_sequences=True)(input_data)
    lstm_layer2 = LSTM(20)(lstm_layer1)
    x = Flatten()(lstm_layer2)
    output_data = Dense(1)(x)
    model = Model(input_data, output_data)
    return model

## 5. Display Model Summary and Diagram
The model summary and diagram verify the LSTM architecture, input shape, and trainable parameters.


In [5]:
model1 = create_lstm()
model1.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 24, 21)]          0         
                                                                 
 lstm (LSTM)                 (None, 24, 8)             960       
                                                                 
 lstm_1 (LSTM)               (None, 20)                2320      
                                                                 
 flatten (Flatten)           (None, 20)                0         
                                                                 
 dense (Dense)               (None, 1)                 21        
                                                                 
Total params: 3,301
Trainable params: 3,301
Non-trainable params: 0
_________________________________________________________________


In [6]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.


## 6. Set Checkpoint and History Paths
These paths define where the best LSTM model and training history will be saved.


In [7]:
checkpoints = r'C:\Users\Admin\Downloads\labs\New folder\Code\\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'C:\Users\Admin\Downloads\labs\New folder\Code'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

## 7. Configure Callbacks
The checkpoint callback saves the best validation-loss model, and the training monitor records training progress.


In [8]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

## 8. Compile or Load the LSTM
A new LSTM is compiled when no model checkpoint is supplied. Otherwise, the saved model is loaded and training can continue.


In [9]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =create_lstm()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


## 9. Load Dataset Files
Training, validation, and testing CSV files are loaded from the processed dataset folder. The scaler is loaded for converting predictions back to original units.


In [10]:
import os

print(os.getcwd())

C:\Users\Admin\Downloads\labs\New folder\Code


In [13]:
import os
path_dataset = r"C:\Users\Admin\Downloads\labs\New folder"
path_tr = os.path.join(path_dataset, 'AEP_train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'AEP_validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'AEP_test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEPscaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\Admin\anaconda3\envs\ML\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((84907, 21), (24259, 21), (12130, 21))

## 10. Prepare Sequential Data
The time-step configuration is confirmed, then the dataset is converted into supervised sequence windows for the LSTM.


In [14]:
time_steps=24
num_features=21

In [15]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 1.4650800228118896 sec


## 11. Train the LSTM
The LSTM is trained on sequential windows and validated after each epoch. Checkpoints are saved based on validation loss.


In [16]:
epochs = 3
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,verbose = verbose)

Epoch 1/3
2653/2653 [==============================] - ETA: 0s - loss: 0.0492 - mae: 0.0492 - mape: 6511.2358
Epoch 1: val_loss improved from inf to 0.02800, saving model to C:\Users\Admin\Downloads\labs\New folder\Code\E1-cp-0001-loss0.03.h5
2653/2653 [==============================] - 114s 39ms/step - loss: 0.0492 - mae: 0.0492 - mape: 6511.2358 - val_loss: 0.0280 - val_mae: 0.0280 - val_mape: 9.7896
Epoch 2/3
2652/2653 [============================>.] - ETA: 0s - loss: 0.0234 - mae: 0.0234 - mape: 222.6477
Epoch 2: val_loss improved from 0.02800 to 0.01928, saving model to C:\Users\Admin\Downloads\labs\New folder\Code\E1-cp-0002-loss0.02.h5
2653/2653 [==============================] - 105s 39ms/step - loss: 0.0234 - mae: 0.0234 - mape: 222.6017 - val_loss: 0.0193 - val_mae: 0.0193 - val_mape: 7.0827
Epoch 3/3
2653/2653 [==============================] - ETA: 0s - loss: 0.0156 - mae: 0.0156 - mape: 30.4763
Epoch 3: val_loss improved from 0.01928 to 0.01264, saving model to C:\Users\A

## 12. Evaluate Test Performance
The best saved model is loaded, test predictions are inverse-transformed, and MAE, RMSE, MAPE, and R2 are calculated.


In [18]:

model = load_model(r'C:\Users\Admin\Downloads\labs\New folder\Code\E1-cp-0010-loss0.04.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

379/379 [==============================] - 2s 5ms/step
Mean Absolute Error (MAE): 1928.76
Median Absolute Error (MedAE): 1568.13
Mean Squared Error (MSE): 6048747.17
Root Mean Squared Error (RMSE): 2459.42
Mean Absolute Percentage Error (MAPE): 13.42 %
Median Absolute Percentage Error (MDAPE): 10.51 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)


## 13. Fine-Tuning Setup
This cell prepares a second training stage by selecting a previous model checkpoint and setting the starting epoch.


In [19]:
checkpoints = r'C:\Users\Admin\Downloads\labs\New folder\Code\E1-cp-0010-loss0.04.h5'
model=r'C:\Users\Admin\Downloads\labs\New folder\Code\E1-cp-0010-loss0.04.h5'
start_epoch= 3

## 14. Rebuild Callbacks and Load Model
Callbacks and the model are prepared again so fine-tuning can continue from the saved checkpoint.


In [20]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] loading C:\Users\Admin\Downloads\labs\New folder\Code\E1-cp-0010-loss0.04.h5...
[INFO] old learning rate: 0.0010000000474974513
[INFO] new learning rate: 9.999999747378752e-05


## 15. Continue Training
The LSTM is trained for additional epochs to refine its forecasting performance.


In [22]:
epochs = 3
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/3
2652/2653 [============================>.] - ETA: 0s - loss: 0.0463 - mae: 0.0463 - mape: 1129.9871
Epoch 1: val_loss improved from inf to 0.04288, saving model to C:\Users\Admin\Downloads\labs\New folder\Code\E1-cp-0010-loss0.04.h5
2653/2653 [==============================] - 20s 8ms/step - loss: 0.0463 - mae: 0.0463 - mape: 1129.7494 - val_loss: 0.0429 - val_mae: 0.0429 - val_mape: 15.3993
Epoch 2/3
2650/2653 [============================>.] - ETA: 0s - loss: 0.0394 - mae: 0.0394 - mape: 553.4027
Epoch 2: val_loss improved from 0.04288 to 0.03760, saving model to C:\Users\Admin\Downloads\labs\New folder\Code\E1-cp-0010-loss0.04.h5
2653/2653 [==============================] - 19s 7ms/step - loss: 0.0394 - mae: 0.0394 - mape: 552.8780 - val_loss: 0.0376 - val_mae: 0.0376 - val_mape: 13.3803
Epoch 3/3
2645/2653 [============================>.] - ETA: 0s - loss: 0.0348 - mae: 0.0348 - mape: 449.6093
Epoch 3: val_loss improved from 0.03760 to 0.03328, saving model to C:\Users\Ad

## 16. Evaluate Fine-Tuned LSTM
The fine-tuned model is evaluated on the test set using the same metrics for a fair comparison.


In [23]:

model = load_model(r'C:\Users\Admin\Downloads\labs\New folder\Code\E1-cp-0010-loss0.04.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

379/379 [==============================] - 3s 6ms/step
Mean Absolute Error (MAE): 415.83
Median Absolute Error (MedAE): 341.39
Mean Squared Error (MSE): 284028.31
Root Mean Squared Error (RMSE): 532.94
Mean Absolute Percentage Error (MAPE): 2.84 %
Median Absolute Percentage Error (MDAPE): 2.36 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)


## Conclusion
This lab demonstrates the use of LSTM networks for time-series forecasting. LSTMs are suitable for sequential data because they can learn dependencies across previous time steps, and fine-tuning helps improve a saved model.
